# CNN Model Tutorial

This tutorial covers how to use `CNNModel` in ALF for **regression** and **classification** tasks on protein sequences.

> **Prerequisites:** Familiarity with the ALF offline design loop.  
> See [offline_design_tutorial.ipynb](../experiments/offline_design_tutorial.ipynb) for that foundation.

### What you'll learn
1. How to configure and train `CNNModel` for regression (continuous output)
2. How to switch to binary classification (two phenotype classes)
3. How to switch to multiclass classification (three or more classes)
4. Which metrics are reported for each `ProblemType`

### Environment Setup

From the `/tutorials` directory:
```
uv sync
source .venv/bin/activate
```
Select `.venv` as the kernel when prompted.

In [ ]:
import numpy as np
from alf_core.dataclasses.candidate import Candidate, Modality
from alf_core.dataclasses.labelled_candidates import LabelledCandidates
from alf_core.dataset.base_dataset import BaseDataset, BaseDatasetConfig
from alf_core.utils.enums import ProblemType
from alf_tools.models.cnn import CNNModel, CNNModelConfig, CNNTrainConfig

print('Imports successful')

---
## Section 1: Regression

We use synthetic sequences with continuous fitness scores.  
`problem_type=ProblemType.REGRESSION` → single output neuron, MSE/Pearson/Spearman metrics.

In [ ]:
ALPHABET = list('ACDEFGHIKLMNPQRSTVWY')
SEQ_LEN = 20
rng = np.random.default_rng(42)

def random_sequences(n: int) -> list[str]:
    return [''.join(rng.choice(ALPHABET, SEQ_LEN)) for _ in range(n)]

def make_lc(seqs: list[str], labels: np.ndarray) -> LabelledCandidates:
    candidates = [Candidate(data=s, modality=Modality.SEQUENCE) for s in seqs]
    return LabelledCandidates(candidates=candidates, labels=labels)


seqs_reg = random_sequences(200)
labels_reg = rng.normal(loc=0.5, scale=0.2, size=200).clip(0, 1).astype(np.float32)

print(f'Sequences: {len(seqs_reg)}')
print(f'Label range: [{labels_reg.min():.3f}, {labels_reg.max():.3f}]')

In [ ]:
class SyntheticDataset(BaseDataset):
    def __init__(self, config: BaseDatasetConfig, lc: LabelledCandidates):
        super().__init__(config)
        self._lc = lc

    def load_dataset(self) -> LabelledCandidates:
        return self._lc


reg_config = BaseDatasetConfig(
    name='synthetic_regression',
    modality=Modality.SEQUENCE,
    seed=42,
    train_ratio=0.8,
    validation_frac=0.2,
    test_ratio=0.2,
    split_type='random',
    problem_type=ProblemType.REGRESSION,
)

reg_dataset = SyntheticDataset(reg_config, make_lc(seqs_reg, labels_reg))
reg_dataset.setup()

cnn_regression = CNNModel(
    name='cnn_regression',
    model_config=CNNModelConfig(num_filters=32, kernel_size=3, num_conv_layers=2, fc_hidden_dim=64),
    train_config=CNNTrainConfig(num_epochs=5, batch_size=32, learning_rate=1e-3, log_frequency=5),
)

cnn_regression.setup(reg_dataset)
cnn_regression.train(
    train_data=reg_dataset.train_dataset,
    val_data=reg_dataset.validation_dataset,
)

print('Training complete.')
metrics = cnn_regression.get_training_summary_metrics()
print('Train metrics:', {k: f'{v:.4f}' for k, v in metrics.items()})

In [ ]:
val_reg = reg_dataset.validation_dataset
preds = cnn_regression.predict(val_reg.candidates)
print(f'predictions.means shape: {preds.means.shape}')   # (n,)
print(f'predictions.variances:   {preds.variances}')      # None
print(f'Sample outputs: {preds.means[:5].round(4)}')

---
## Section 2: Binary Classification

Binary labels must be integers `{0, 1}`.  
`problem_type=ProblemType.BINARY` → sigmoid activation → `predictions.means` shape `(n, 2)` (class probabilities).  
Metrics reported: `accuracy`, `f1`, `precision`, `recall`, `auc_roc`.

In [ ]:
seqs_bin = random_sequences(200)
labels_bin = (rng.random(200) > 0.5).astype(np.int32)

bin_config = BaseDatasetConfig(
    name='synthetic_binary',
    modality=Modality.SEQUENCE,
    seed=42,
    train_ratio=0.8,
    validation_frac=0.2,
    test_ratio=0.2,
    split_type='stratified',  # preserves class balance across splits
    problem_type=ProblemType.BINARY,
)

bin_dataset = SyntheticDataset(bin_config, make_lc(seqs_bin, labels_bin))
bin_dataset.setup()

print(f'num_classes: {bin_dataset.num_classes}')  # 2
print(f'Train: {len(bin_dataset.train_dataset)}, Val: {len(bin_dataset.validation_dataset)}')

In [ ]:
cnn_binary = CNNModel(
    name='cnn_binary',
    model_config=CNNModelConfig(num_filters=32, kernel_size=3, num_conv_layers=2, fc_hidden_dim=64),
    train_config=CNNTrainConfig(num_epochs=5, batch_size=32, learning_rate=1e-3, log_frequency=5),
)

cnn_binary.setup(bin_dataset)
cnn_binary.train(
    train_data=bin_dataset.train_dataset,
    val_data=bin_dataset.validation_dataset,
)

print('Training complete.')
metrics = cnn_binary.get_training_summary_metrics()
print('Train metrics:', {k: f'{v:.4f}' for k, v in metrics.items()})

In [ ]:
val_bin = bin_dataset.validation_dataset
preds = cnn_binary.predict(val_bin.candidates)
print(f'predictions.means shape: {preds.means.shape}')  # (n, 2)
print(f'Sample class probabilities (first 3 rows):')
print(preds.means[:3].round(4))
print(f'Predicted classes: {preds.means[:3].argmax(axis=1)}')

---
## Section 3: Multiclass Classification

Multiclass labels must be integers `{0, 1, ..., K-1}` with at least 3 distinct classes.  
`problem_type=ProblemType.MULTICLASS` → softmax activation → `predictions.means` shape `(n, K)`.  
Metrics reported: `accuracy`, `f1` (macro), `precision` (macro), `recall` (macro), `auc_roc` (OvR).

In [ ]:
seqs_mc = random_sequences(300)
labels_mc = rng.integers(0, 4, size=300).astype(np.int32)  # 4 classes

mc_config = BaseDatasetConfig(
    name='synthetic_multiclass',
    modality=Modality.SEQUENCE,
    seed=42,
    train_ratio=0.8,
    validation_frac=0.2,
    test_ratio=0.2,
    split_type='stratified',
    problem_type=ProblemType.MULTICLASS,
)

mc_dataset = SyntheticDataset(mc_config, make_lc(seqs_mc, labels_mc))
mc_dataset.setup()

print(f'num_classes: {mc_dataset.num_classes}')  # 4

In [ ]:
cnn_multiclass = CNNModel(
    name='cnn_multiclass',
    model_config=CNNModelConfig(num_filters=32, kernel_size=3, num_conv_layers=2, fc_hidden_dim=64),
    train_config=CNNTrainConfig(num_epochs=5, batch_size=32, learning_rate=1e-3, log_frequency=5),
)

cnn_multiclass.setup(mc_dataset)
cnn_multiclass.train(
    train_data=mc_dataset.train_dataset,
    val_data=mc_dataset.validation_dataset,
)

print('Training complete.')
metrics = cnn_multiclass.get_training_summary_metrics()
print('Train metrics:', {k: f'{v:.4f}' for k, v in metrics.items()})

In [ ]:
val_mc = mc_dataset.validation_dataset
preds = cnn_multiclass.predict(val_mc.candidates)
print(f'predictions.means shape: {preds.means.shape}')  # (n, 4)
print(f'Rows sum to 1 (softmax check): {preds.means[:3].sum(axis=1).round(4)}')
print(f'Predicted classes: {preds.means[:5].argmax(axis=1)}')

---
## Summary

| `problem_type` | Labels | Output shape | Activation | Metrics |
|---|---|---|---|---|
| `REGRESSION` | `float32` | `(n,)` | identity | MSE, Pearson, Spearman, ECE, … |
| `BINARY` | `int {0,1}` | `(n, 2)` | sigmoid → complement pair | accuracy, f1, precision, recall, auc_roc |
| `MULTICLASS` | `int {0…K-1}` | `(n, K)` | softmax | accuracy, f1 (macro), precision, recall, auc_roc (OvR) |

**Key points:**
- `problem_type` is set on `BaseDatasetConfig`, not on `CNNModel` directly — the model reads it from the dataset.
- `num_classes` is derived automatically from unique integer labels in the dataset.
- Use `split_type='stratified'` for classification to preserve class balance.
- `CNNModelConfig` is the same for all problem types — architecture does not change, only the output layer size and activation.